In [ ]:
import os
import re
import csv
import json
import pickle
import pandas as pd
import numpy as np
from pathlib import Path
from google.colab import drive

In [ ]:
WINDOW_SIZE = "30min"
MIN_SEQ_LEN = 5
MAX_CHUNK_LEN = 64
STRIDE = 32

VAL_B_SIZE = 0.50
TEST_B_SIZE = 0.50

SERVER_NAME = "payment_server_B"
SERVER_B_START_DATE = "2026-04-04"

FORCE_RECREATE_LOCKED_SPLIT = True

SAVE_DIR = "/content/drive/MyDrive/windows_logs_project"
VERSION = "serverB_payment_30min_val_test"



In [ ]:
drive.mount("/content/drive")

os.makedirs(SAVE_DIR, exist_ok=True)

LOCKED_DATA_PATH = f"{SAVE_DIR}/payment_serverB_locked_val_test_30min.pkl"
EXPORT_DIR = f"{SAVE_DIR}/payment_serverB_export_30min"
PROMPT_EXPORT_DIR = f"{SAVE_DIR}/payment_serverB_annotation_prompts_30min"

os.makedirs(EXPORT_DIR, exist_ok=True)
os.makedirs(PROMPT_EXPORT_DIR, exist_ok=True)

print("SAVE_DIR:", SAVE_DIR)
print("LOCKED_DATA_PATH:", LOCKED_DATA_PATH)
print("EXPORT_DIR:", EXPORT_DIR)
print("PROMPT_EXPORT_DIR:", PROMPT_EXPORT_DIR)

Mounted at /content/drive
SAVE_DIR: /content/drive/MyDrive/windows_logs_project
LOCKED_DATA_PATH: /content/drive/MyDrive/windows_logs_project/payment_serverB_locked_val_test_30min.pkl
EXPORT_DIR: /content/drive/MyDrive/windows_logs_project/payment_serverB_export_30min
PROMPT_EXPORT_DIR: /content/drive/MyDrive/windows_logs_project/payment_serverB_annotation_prompts_30min


In [ ]:
WINDOWS_COLUMNS = [
    "Level",
    "Date and Time",
    "Source",
    "Event ID",
    "Task Category",
    "Message"
]


def load_windows_event_csv(file_path: str, log_name: str) -> pd.DataFrame:
    rows = []

    with open(file_path, "r", encoding="utf-8-sig", errors="replace", newline="") as f:
        reader = csv.reader(f)
        broken_header = next(reader, None)

        print(f"{log_name} | {Path(file_path).name} исходный header:", broken_header)

        for row in reader:
            if not row or all(not str(x).strip() for x in row):
                continue

            if len(row) == 6:
                rows.append(row)
            elif len(row) > 6:
                rows.append(row[:5] + [",".join(row[5:])])
            else:
                rows.append(row + [None] * (6 - len(row)))

    df = pd.DataFrame(rows, columns=WINDOWS_COLUMNS)
    df["LogName"] = log_name
    df["SourceFile"] = Path(file_path).name

    return df


def clean_windows_events(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    for col in ["Level", "Date and Time", "Source", "Event ID", "Task Category"]:
        df[col] = df[col].astype(str).str.strip()

    df["Message"] = df["Message"].fillna("").astype(str).str.strip()

    df["timestamp"] = pd.to_datetime(
        df["Date and Time"],
        errors="coerce",
        dayfirst=True
    )

    before = len(df)
    df = df.dropna(subset=["timestamp"]).copy()

    print(f"Удалено строк с некорректной датой: {before - len(df)}")

    return df


def load_many_windows_logs(files_by_logname: dict) -> pd.DataFrame:
    all_parts = []

    for log_name, paths in files_by_logname.items():
        for path in paths:
            raw = load_windows_event_csv(path, log_name)
            clean = clean_windows_events(raw)
            all_parts.append(clean)

    all_logs = pd.concat(all_parts, ignore_index=True)

    print("До удаления дубликатов:", len(all_logs))

    dedup_cols = [
        "LogName",
        "timestamp",
        "Level",
        "Source",
        "Event ID",
        "Task Category",
        "Message"
    ]

    duplicate_count = all_logs.duplicated(subset=dedup_cols).sum()
    print("Количество дубликатов:", duplicate_count)

    all_logs = all_logs.drop_duplicates(subset=dedup_cols, keep="first").copy()

    print("После удаления дубликатов:", len(all_logs))

    all_logs = all_logs.sort_values("timestamp").reset_index(drop=True)

    print("Диапазон дат:", all_logs["timestamp"].min(), "—", all_logs["timestamp"].max())

    return all_logs

In [ ]:
files_by_logname = {
    "System": [
        "/content/payment sys.csv"
    ],
    "Application": [
        "/content/payment app.csv"
    ]
}

windows_clean = load_many_windows_logs(files_by_logname)

windows_clean["Server"] = SERVER_NAME

print("До фильтрации по дате:", windows_clean.shape)
print("Диапазон дат:", windows_clean["timestamp"].min(), "—", windows_clean["timestamp"].max())

windows_clean = windows_clean[
    windows_clean["timestamp"] >= pd.Timestamp(SERVER_B_START_DATE)
].copy()

windows_clean = windows_clean.sort_values("timestamp").reset_index(drop=True)

print("После фильтрации по дате:", windows_clean.shape)
print("Диапазон дат:", windows_clean["timestamp"].min(), "—", windows_clean["timestamp"].max())

print("\nРаспределение Level:")
print(windows_clean["Level"].value_counts(dropna=False))
print(windows_clean["Level"].value_counts(normalize=True, dropna=False))

System | payment sys.csv исходный header: ['Level', 'Date and Time', 'Source', 'Event ID', 'Task Category']
Удалено строк с некорректной датой: 0
Application | payment app.csv исходный header: ['Level', 'Date and Time', 'Source', 'Event ID', 'Task Category']
Удалено строк с некорректной датой: 0
До удаления дубликатов: 41620
Количество дубликатов: 165
После удаления дубликатов: 41455
Диапазон дат: 2025-09-07 23:07:55 — 2026-05-05 07:48:22
До фильтрации по дате: (41455, 10)
Диапазон дат: 2025-09-07 23:07:55 — 2026-05-05 07:48:22
После фильтрации по дате: (34195, 10)
Диапазон дат: 2026-04-04 00:00:16 — 2026-05-05 07:48:22

Распределение Level:
Level
Information    33722
Warning          465
Error              8
Name: count, dtype: int64
Level
Information    0.986168
Warning        0.013598
Error          0.000234
Name: proportion, dtype: float64


In [ ]:
def normalize_message(text: str) -> str:
    text = str(text).lower()

    # GUID
    text = re.sub(
        r"\b[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}\b",
        "<guid>",
        text
    )

    # IP addresses
    text = re.sub(
        r"\b(?:\d{1,3}\.){3}\d{1,3}\b",
        "<ip>",
        text
    )

    # Windows paths
    text = re.sub(
        r"[a-z]:\\(?:[^\\/:*?\"<>|\r\n]+\\)*[^\\/:*?\"<>|\r\n]*",
        "<path>",
        text
    )

    # Dates
    text = re.sub(
        r"\b\d{1,2}[./-]\d{1,2}[./-]\d{2,4}\b",
        "<date>",
        text
    )

    # Time
    text = re.sub(
        r"\b\d{1,2}:\d{2}(:\d{2})?\b",
        "<time>",
        text
    )

    # Hex values
    text = re.sub(
        r"\b0x[0-9a-f]+\b",
        "<hex>",
        text
    )

    # Numbers
    text = re.sub(
        r"\b\d+\b",
        "<num>",
        text
    )

    # Extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
windows_clean["NormalizedMessage"] = windows_clean["Message"].apply(normalize_message)

display(windows_clean[[
    "timestamp",
    "LogName",
    "Level",
    "Source",
    "Event ID",
    "Task Category",
    "Message",
    "NormalizedMessage"
]].head(10))

,timestamp,LogName,Level,Source,Event ID,Task Category,Message,NormalizedMessage
0,2026-04-04 00:00:16,Application,Information,MSSQLSERVER,17177,Server,This instance of SQL Server has been using a p...,this instance of sql server has been using a p...
1,2026-04-04 00:49:06,Application,Information,VSS,8224,None,The VSS service is shutting down due to idle t...,the vss service is shutting down due to idle t...
2,2026-04-04 01:11:14,Application,Warning,SceCli,1202,None,Security policies were propagated with warning...,security policies were propagated with warning...
3,2026-04-04 03:00:22,Application,Warning,SceCli,1202,None,Security policies were propagated with warning...,security policies were propagated with warning...
4,2026-04-04 03:43:26,Application,Information,Microsoft-Windows-Security-SPP,16394,None,Offline downlevel migration succeeded.,offline downlevel migration succeeded.
5,2026-04-04 03:43:59,Application,Information,Microsoft-Windows-Security-SPP,16384,None,Successfully scheduled Software Protection ser...,successfully scheduled software protection ser...
6,2026-04-04 04:06:15,Application,Warning,Microsoft-Windows-Perflib,1008,None,"The Open procedure for service ""BITS"" in DLL ""...","the open procedure for service ""bits"" in dll ""..."
7,2026-04-04 04:38:30,Application,Warning,SceCli,1202,None,Security policies were propagated with warning...,security policies were propagated with warning...
8,2026-04-04 06:25:36,Application,Warning,SceCli,1202,None,Security policies were propagated with warning...,security policies were propagated with warning...
9,2026-04-04 06:35:56,Application,Information,Microsoft-Windows-Security-SPP,16394,None,Offline downlevel migration succeeded.,offline downlevel migration succeeded.


In [ ]:
windows_clean["EventToken"] = windows_clean.apply(
    lambda row: (
        f"{row['LogName']} | "
        f"{row['Source']} | "
        f"{row['Event ID']} | "
        f"{row['Level']} | "
        f"{row['NormalizedMessage']}"
    ),
    axis=1
)

windows_clean["EventTemplateForReview"] = windows_clean["EventToken"]

print("Уникальных EventToken:", windows_clean["EventToken"].nunique())

display(windows_clean[[
    "timestamp",
    "LogName",
    "Level",
    "Source",
    "Event ID",
    "EventToken"
]].head(10))

Уникальных EventToken: 203


,timestamp,LogName,Level,Source,Event ID,EventToken
0,2026-04-04 00:00:16,Application,Information,MSSQLSERVER,17177,Application | MSSQLSERVER | 17177 | Informatio...
1,2026-04-04 00:49:06,Application,Information,VSS,8224,Application | VSS | 8224 | Information | the v...
2,2026-04-04 01:11:14,Application,Warning,SceCli,1202,Application | SceCli | 1202 | Warning | securi...
3,2026-04-04 03:00:22,Application,Warning,SceCli,1202,Application | SceCli | 1202 | Warning | securi...
4,2026-04-04 03:43:26,Application,Information,Microsoft-Windows-Security-SPP,16394,Application | Microsoft-Windows-Security-SPP |...
5,2026-04-04 03:43:59,Application,Information,Microsoft-Windows-Security-SPP,16384,Application | Microsoft-Windows-Security-SPP |...
6,2026-04-04 04:06:15,Application,Warning,Microsoft-Windows-Perflib,1008,Application | Microsoft-Windows-Perflib | 1008...
7,2026-04-04 04:38:30,Application,Warning,SceCli,1202,Application | SceCli | 1202 | Warning | securi...
8,2026-04-04 06:25:36,Application,Warning,SceCli,1202,Application | SceCli | 1202 | Warning | securi...
9,2026-04-04 06:35:56,Application,Information,Microsoft-Windows-Security-SPP,16394,Application | Microsoft-Windows-Security-SPP |...


In [ ]:
unique_tokens = (
    windows_clean["EventToken"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

template_to_id = {
    token: f"W{i + 1}"
    for i, token in enumerate(unique_tokens)
}

id_to_template = {
    v: k
    for k, v in template_to_id.items()
}

windows_clean["EventTemplateID"] = windows_clean["EventToken"].map(template_to_id)

print("Количество шаблонов Windows:", len(template_to_id))

display(windows_clean[[
    "timestamp",
    "LogName",
    "Level",
    "Source",
    "Event ID",
    "EventTemplateID",
    "EventToken"
]].head(10))

Количество шаблонов Windows: 203


,timestamp,LogName,Level,Source,Event ID,EventTemplateID,EventToken
0,2026-04-04 00:00:16,Application,Information,MSSQLSERVER,17177,W2,Application | MSSQLSERVER | 17177 | Informatio...
1,2026-04-04 00:49:06,Application,Information,VSS,8224,W140,Application | VSS | 8224 | Information | the v...
2,2026-04-04 01:11:14,Application,Warning,SceCli,1202,W138,Application | SceCli | 1202 | Warning | securi...
3,2026-04-04 03:00:22,Application,Warning,SceCli,1202,W138,Application | SceCli | 1202 | Warning | securi...
4,2026-04-04 03:43:26,Application,Information,Microsoft-Windows-Security-SPP,16394,W130,Application | Microsoft-Windows-Security-SPP |...
5,2026-04-04 03:43:59,Application,Information,Microsoft-Windows-Security-SPP,16384,W54,Application | Microsoft-Windows-Security-SPP |...
6,2026-04-04 04:06:15,Application,Warning,Microsoft-Windows-Perflib,1008,W8,Application | Microsoft-Windows-Perflib | 1008...
7,2026-04-04 04:38:30,Application,Warning,SceCli,1202,W138,Application | SceCli | 1202 | Warning | securi...
8,2026-04-04 06:25:36,Application,Warning,SceCli,1202,W138,Application | SceCli | 1202 | Warning | securi...
9,2026-04-04 06:35:56,Application,Information,Microsoft-Windows-Security-SPP,16394,W130,Application | Microsoft-Windows-Security-SPP |...


In [ ]:
HARD_CRITICAL_EVENT_IDS = {
    "41",
    "6008",
    "7031",
    "7034",
    "17832",
    "17883",
    "1102",
    "4625",
    "4740",
    "9002",
    "4014",
    "18456"
}

SOFT_SUSPICIOUS_EVENT_IDS = {
    "1000",
    "1001",
    "10010",
    "10036",
    "1801",
    "1202",
    "1065",
    "8193"
}

Временные окна

In [ ]:
def build_sequences_by_time_window(
    df: pd.DataFrame,
    window: str = "30min",
    min_seq_len: int = 5
) -> pd.DataFrame:
    df = df.copy()
    df = df.sort_values("timestamp").reset_index(drop=True)

    df["window_start"] = df["timestamp"].dt.floor(window)

    sequences = []

    for window_start, group in df.groupby("window_start"):
        group = group.sort_values("timestamp")

        features = group["EventTemplateID"].tolist()
        timestamps = group["timestamp"].tolist()

        if len(features) < min_seq_len:
            continue

        time_intervals = [0.0]

        for i in range(1, len(timestamps)):
            delta = (timestamps[i] - timestamps[i - 1]).total_seconds()
            time_intervals.append(float(delta))

        latency = float((timestamps[-1] - timestamps[0]).total_seconds())

        sequences.append({
            "WindowSize": window,
            "WindowStart": window_start,
            "WindowEnd": window_start + pd.Timedelta(window),

            "Features": features,
            "TimeInterval": time_intervals,
            "Latency": latency,
            "SeqLen": len(features),


            "LogNames": sorted(group["LogName"].astype(str).unique().tolist()),
            "Sources": sorted(group["Source"].astype(str).unique().tolist()),
            "Levels": sorted(group["Level"].astype(str).unique().tolist()),
            "EventIDs": sorted(group["Event ID"].astype(str).unique().tolist()),

            "EventLogNames": group["LogName"].astype(str).tolist(),
            "EventSources": group["Source"].astype(str).tolist(),
            "EventLevels": group["Level"].astype(str).tolist(),
            "EventIDsPerEvent": group["Event ID"].astype(str).tolist(),
            "EventTokens": group["EventToken"].astype(str).tolist(),

            "RawMessages": group["Message"].astype(str).tolist(),
            "ReviewTemplates": group["EventTemplateForReview"].astype(str).tolist(),

            "Label": None
        })

    return pd.DataFrame(sequences)

In [ ]:
def recompute_weak_label_for_row(row, q95_len=None):
    levels = [str(x).lower() for x in row["Levels"]]
    event_ids = set(str(x) for x in row["EventIDs"])
    sources = [str(x).lower() for x in row["Sources"]]

    hard_reasons = []
    soft_reasons = []

    if any("critical" in x for x in levels):
        hard_reasons.append("critical_level")

    matched_hard_ids = event_ids.intersection(HARD_CRITICAL_EVENT_IDS)
    if matched_hard_ids:
        hard_reasons.append(
            "hard_critical_event_id:" + ",".join(sorted(matched_hard_ids))
        )

    if any("mssqlserver" in s for s in sources) and any("error" in x for x in levels):
        hard_reasons.append("mssql_error")

    if any("error" in x for x in levels):
        soft_reasons.append("error_level")

    matched_soft_ids = event_ids.intersection(SOFT_SUSPICIOUS_EVENT_IDS)
    if matched_soft_ids:
        soft_reasons.append(
            "soft_suspicious_event_id:" + ",".join(sorted(matched_soft_ids))
        )

    if q95_len is not None and row["SeqLen"] > q95_len:
        soft_reasons.append("high_event_volume")

    weak_label = int(len(hard_reasons) > 0 or len(soft_reasons) > 0)
    hard_weak_label = int(len(hard_reasons) > 0)

    row["WeakLabel"] = weak_label
    row["HardWeakLabel"] = hard_weak_label
    row["HardSuspiciousReason"] = "; ".join(hard_reasons)
    row["SoftSuspiciousReason"] = "; ".join(soft_reasons)
    row["SuspiciousReason"] = "; ".join(hard_reasons + soft_reasons)
    row["NeedsReview"] = weak_label == 1

    return row


def add_weak_labels(windows_seq: pd.DataFrame) -> pd.DataFrame:
    windows_seq = windows_seq.copy()

    q95_len = windows_seq["SeqLen"].quantile(0.95)

    rows = []

    for _, row in windows_seq.iterrows():
        rows.append(recompute_weak_label_for_row(row.copy(), q95_len=q95_len))

    return pd.DataFrame(rows).reset_index(drop=True)

In [ ]:
windows_seq_15 = build_sequences_by_time_window(
    windows_clean,
    window="15min",
    min_seq_len=MIN_SEQ_LEN
)

windows_seq_30 = build_sequences_by_time_window(
    windows_clean,
    window="30min",
    min_seq_len=MIN_SEQ_LEN
)

windows_seq_15 = add_weak_labels(windows_seq_15)
windows_seq_30 = add_weak_labels(windows_seq_30)

print("15 минут:", windows_seq_15.shape)
print("30 минут:", windows_seq_30.shape)

15 минут: (2923, 25)
30 минут: (1462, 25)


In [ ]:
def summarize_windows_sequences(name, seq_df):
    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    print("Количество окон:", len(seq_df))
    print("Диапазон времени:", seq_df["WindowStart"].min(), "—", seq_df["WindowEnd"].max())

    print("\nДлины последовательностей:")
    display(seq_df["SeqLen"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

    print("\nWeakLabel distribution:")
    print(seq_df["WeakLabel"].value_counts(dropna=False))
    print(seq_df["WeakLabel"].value_counts(normalize=True, dropna=False))

    print("\nТоп-10 самых длинных окон:")
    display(
        seq_df.sort_values("SeqLen", ascending=False).head(10)[[
            "WindowStart",
            "WindowEnd",
            "SeqLen",
            "Levels",
            "EventIDs",
            "WeakLabel",
            "SuspiciousReason"
        ]]
    )


summarize_windows_sequences("Окна 15 минут", windows_seq_15)
summarize_windows_sequences("Окна 30 минут", windows_seq_30)


Окна 15 минут
Количество окон: 2923
Диапазон времени: 2026-04-04 21:15:00 — 2026-05-05 08:00:00

Длины последовательностей:


,SeqLen
count,2923.000000
mean,11.689360
std,5.274204
min,5.000000
50%,10.000000
75%,13.000000
90%,21.000000
95%,23.000000
99%,27.000000
max,38.000000



WeakLabel distribution:
WeakLabel
0    2489
1     434
Name: count, dtype: int64
WeakLabel
0    0.851522
1    0.148478
Name: proportion, dtype: float64

Топ-10 самых длинных окон:


,WindowStart,WindowEnd,SeqLen,Levels,EventIDs,WeakLabel,SuspiciousReason
421,2026-04-09 06:30:00,2026-04-09 06:45:00,38,"[Information, Warning]","[1202, 1502, 16384, 16394, 7036, 7040]",1,soft_suspicious_event_id:1202; high_event_volume
991,2026-04-15 05:00:00,2026-04-15 05:15:00,35,"[Information, Warning]","[1202, 1502, 16384, 16394, 7036, 7040]",1,soft_suspicious_event_id:1202; high_event_volume
349,2026-04-08 12:30:00,2026-04-08 12:45:00,34,"[Information, Warning]","[1202, 1502, 16384, 16394, 7036, 7040]",1,soft_suspicious_event_id:1202; high_event_volume
2368,2026-04-29 13:15:00,2026-04-29 13:30:00,32,"[Information, Warning]","[1202, 1502, 16384, 16394, 7036, 7040]",1,soft_suspicious_event_id:1202; high_event_volume
1383,2026-04-19 07:00:00,2026-04-19 07:15:00,31,"[Information, Warning]","[1202, 1502, 16394, 7036, 7040]",1,soft_suspicious_event_id:1202; high_event_volume
219,2026-04-07 04:00:00,2026-04-07 04:15:00,31,"[Information, Warning]","[1008, 1202, 1502, 16384, 16394, 7036, 7040]",1,soft_suspicious_event_id:1202; high_event_volume
1695,2026-04-22 13:00:00,2026-04-22 13:15:00,30,"[Error, Information]","[1003, 1033, 1034, 12308, 16384, 16394, 7036, ...",1,error_level; high_event_volume
1957,2026-04-25 06:30:00,2026-04-25 06:45:00,30,[Information],"[16384, 16394, 7036, 7040]",1,high_event_volume
595,2026-04-11 02:00:00,2026-04-11 02:15:00,29,[Information],"[16384, 16394, 7036, 7040]",1,high_event_volume
698,2026-04-12 03:45:00,2026-04-12 04:00:00,29,"[Information, Warning]","[1202, 1502, 16384, 16394, 7036, 7040]",1,soft_suspicious_event_id:1202; high_event_volume



Окна 30 минут
Количество окон: 1462
Диапазон времени: 2026-04-04 21:00:00 — 2026-05-05 08:00:00

Длины последовательностей:


,SeqLen
count,1462.000000
mean,23.372093
std,7.207855
min,9.000000
50%,21.000000
75%,29.000000
90%,34.000000
95%,36.000000
99%,44.000000
max,57.000000



WeakLabel distribution:
WeakLabel
0    1033
1     429
Name: count, dtype: int64
WeakLabel
0    0.706566
1    0.293434
Name: proportion, dtype: float64

Топ-10 самых длинных окон:


,WindowStart,WindowEnd,SeqLen,Levels,EventIDs,WeakLabel,SuspiciousReason
1184,2026-04-29 13:00:00,2026-04-29 13:30:00,57,"[Error, Information, Warning]","[1003, 1033, 1034, 1202, 12308, 1502, 16384, 1...",1,error_level; soft_suspicious_event_id:1202; hi...
1123,2026-04-28 06:30:00,2026-04-28 07:00:00,50,"[Information, Warning]","[1202, 1502, 16384, 16394, 7036, 7040]",1,soft_suspicious_event_id:1202; high_event_volume
1027,2026-04-26 06:30:00,2026-04-26 07:00:00,50,"[Information, Warning]","[1202, 1502, 16, 16384, 16394, 7036, 7040]",1,soft_suspicious_event_id:1202; high_event_volume
211,2026-04-09 06:30:00,2026-04-09 07:00:00,49,"[Information, Warning]","[1202, 1502, 16384, 16394, 7036, 7040]",1,soft_suspicious_event_id:1202; high_event_volume
175,2026-04-08 12:30:00,2026-04-08 13:00:00,49,"[Information, Warning]","[1202, 1502, 16384, 16394, 7036, 7040]",1,soft_suspicious_event_id:1202; high_event_volume
18,2026-04-05 06:00:00,2026-04-05 06:30:00,48,"[Information, Warning]","[1202, 1502, 16384, 16394, 7036, 7040]",1,soft_suspicious_event_id:1202; high_event_volume
1363,2026-05-03 06:30:00,2026-05-03 07:00:00,48,"[Information, Warning]","[1202, 1502, 16, 16384, 16394, 7036, 7040]",1,soft_suspicious_event_id:1202; high_event_volume
67,2026-04-06 06:30:00,2026-04-06 07:00:00,46,"[Information, Warning]","[1202, 1502, 16384, 16394, 7036, 7040]",1,soft_suspicious_event_id:1202; high_event_volume
855,2026-04-22 16:30:00,2026-04-22 17:00:00,46,"[Information, Warning]","[1202, 1502, 16384, 16394, 7036, 7040]",1,soft_suspicious_event_id:1202; high_event_volume
1411,2026-05-04 06:30:00,2026-05-04 07:00:00,46,"[Information, Warning]","[1202, 1502, 16384, 16394, 7036, 7040]",1,soft_suspicious_event_id:1202; high_event_volume


In [ ]:
comparison_rows = []

for name, seq_df in [
    ("15min", windows_seq_15),
    ("30min", windows_seq_30)
]:
    comparison_rows.append({
        "window": name,
        "num_windows": len(seq_df),
        "mean_len": seq_df["SeqLen"].mean(),
        "median_len": seq_df["SeqLen"].median(),
        "p90_len": seq_df["SeqLen"].quantile(0.90),
        "p95_len": seq_df["SeqLen"].quantile(0.95),
        "p99_len": seq_df["SeqLen"].quantile(0.99),
        "max_len": seq_df["SeqLen"].max(),
        "weak_anomaly_count": int(seq_df["WeakLabel"].sum()),
        "weak_anomaly_ratio": seq_df["WeakLabel"].mean()
    })

window_comparison = pd.DataFrame(comparison_rows)

display(window_comparison)

,window,num_windows,mean_len,median_len,p90_len,p95_len,p99_len,max_len,weak_anomaly_count,weak_anomaly_ratio
0,15min,2923,11.689360,10.0,21.0,23.0,27.0,38,434,0.148478
1,30min,1462,23.372093,21.0,34.0,36.0,44.0,57,429,0.293434


In [ ]:
windows_seq = windows_seq_30.copy()

print("Используем окно:", WINDOW_SIZE)
print("windows_seq:", windows_seq.shape)

Используем окно: 30min
windows_seq: (1462, 25)


In [ ]:
def has_level(row, level_name: str) -> bool:
    return any(level_name.lower() in str(x).lower() for x in row["Levels"])


def is_safe_normal(row, q95_len_train) -> bool:
    levels = [str(x).lower() for x in row["Levels"]]
    sources = [str(x).lower() for x in row["Sources"]]
    event_ids = set(str(x) for x in row["EventIDs"])

    has_critical_level = any("critical" in x for x in levels)

    has_hard_critical_event_id = (
        len(event_ids.intersection(HARD_CRITICAL_EVENT_IDS)) > 0
    )

    high_volume = row["SeqLen"] > q95_len_train

    has_error_level = any("error" in x for x in levels)

    serious_error_source = (
        any("mssqlserver" in s for s in sources)
        or any("service control manager" in s for s in sources)
        or any("kernel-power" in s for s in sources)
        or any("eventlog" in s for s in sources)
        or any("sqlserveragent" in s for s in sources)
    )

    has_serious_error = has_error_level and serious_error_source

    return (
        not has_critical_level
        and not has_hard_critical_event_id
        and not high_volume
        and not has_serious_error
    )

In [ ]:
def split_long_sequence_row(row, max_len=64, stride=32, q95_len_for_weak=None):
    n = len(row["Features"])

    per_event_cols = [
        "Features",
        "TimeInterval",
        "RawMessages",
        "ReviewTemplates",
        "EventLogNames",
        "EventSources",
        "EventLevels",
        "EventIDsPerEvent",
        "EventTokens"
    ]

    for col in per_event_cols:
        if col not in row:
            raise ValueError(f"В row отсутствует колонка {col}. Проверь build_sequences_by_time_window.")

    if n <= max_len:
        new_row = row.copy()

        new_row["ChunkID"] = 0
        new_row["OriginalSeqLen"] = n
        new_row["ChunkStartPos"] = 0
        new_row["ChunkEndPos"] = n
        new_row["WasChunked"] = False

        new_row["LogNames"] = sorted(set(map(str, new_row["EventLogNames"])))
        new_row["Sources"] = sorted(set(map(str, new_row["EventSources"])))
        new_row["Levels"] = sorted(set(map(str, new_row["EventLevels"])))
        new_row["EventIDs"] = sorted(set(map(str, new_row["EventIDsPerEvent"])))

        new_row = recompute_weak_label_for_row(
            new_row,
            q95_len=q95_len_for_weak
        )

        return [new_row]

    chunks = []

    starts = list(range(0, n - max_len + 1, stride))
    last_start = n - max_len

    if starts[-1] != last_start:
        starts.append(last_start)

    for chunk_id, start in enumerate(starts):
        end = start + max_len

        new_row = row.copy()

        for col in per_event_cols:
            new_row[col] = row[col][start:end]

        new_row["SeqLen"] = len(new_row["Features"])
        new_row["Latency"] = float(sum(new_row["TimeInterval"]))

        new_row["LogNames"] = sorted(set(map(str, new_row["EventLogNames"])))
        new_row["Sources"] = sorted(set(map(str, new_row["EventSources"])))
        new_row["Levels"] = sorted(set(map(str, new_row["EventLevels"])))
        new_row["EventIDs"] = sorted(set(map(str, new_row["EventIDsPerEvent"])))

        new_row["ChunkID"] = chunk_id
        new_row["OriginalSeqLen"] = n
        new_row["ChunkStartPos"] = start
        new_row["ChunkEndPos"] = end
        new_row["WasChunked"] = True

        new_row = recompute_weak_label_for_row(
            new_row,
            q95_len=q95_len_for_weak
        )

        chunks.append(new_row)

    return chunks


def split_long_sequences(df, max_len=64, stride=32, q95_len_for_weak=None):
    all_rows = []

    for _, row in df.iterrows():
        all_rows.extend(
            split_long_sequence_row(
                row,
                max_len=max_len,
                stride=stride,
                q95_len_for_weak=q95_len_for_weak
            )
        )

    return pd.DataFrame(all_rows).reset_index(drop=True)

In [ ]:
if FORCE_RECREATE_LOCKED_SPLIT and os.path.exists(LOCKED_DATA_PATH):
    os.remove(LOCKED_DATA_PATH)
    print("Удалён старый locked split:", LOCKED_DATA_PATH)


if os.path.exists(LOCKED_DATA_PATH):
    print("Найден locked Server B split. Загружаю существующее разбиение...")

    with open(LOCKED_DATA_PATH, "rb") as f:
        locked_data = pickle.load(f)

    val_seq_raw = locked_data["val_seq_raw"]
    test_seq_raw = locked_data["test_seq_raw"]

    val_seq = locked_data["val_seq"]
    test_seq = locked_data["test_seq"]

    annotation_df = locked_data["annotation_df"]

    template_to_id = locked_data["template_to_id"]
    id_to_template = locked_data["id_to_template"]

    print("Loaded locked Server B data:")
    print("Val raw:", val_seq_raw.shape)
    print("Test raw:", test_seq_raw.shape)
    print("Val chunked:", val_seq.shape)
    print("Test chunked:", test_seq.shape)
    print("Annotation:", annotation_df.shape)

else:
    print("Locked Server B split не найден. Создаю validation_B / test_B...")

    windows_seq_sorted = windows_seq.sort_values("WindowStart").reset_index(drop=True)

    n = len(windows_seq_sorted)
    val_end = int(n * VAL_B_SIZE)

    val_seq_raw = windows_seq_sorted.iloc[:val_end].copy()
    test_seq_raw = windows_seq_sorted.iloc[val_end:].copy()

    val_seq_raw["Split"] = "validation_B"
    test_seq_raw["Split"] = "test_B"

    print("Всего окон Server B:", n)
    print("Validation_B raw:", val_seq_raw.shape)
    print("Test_B raw:", test_seq_raw.shape)

    print("\nДиапазоны времени:")
    print("Validation_B:", val_seq_raw["WindowStart"].min(), "—", val_seq_raw["WindowEnd"].max())
    print("Test_B:", test_seq_raw["WindowStart"].min(), "—", test_seq_raw["WindowEnd"].max())

    assert val_seq_raw["WindowEnd"].max() <= test_seq_raw["WindowStart"].min()

    q95_len_B = windows_seq_sorted["SeqLen"].quantile(0.95)

    print("\n95-й перцентиль длины окон Server B:", q95_len_B)

    val_seq = split_long_sequences(
        val_seq_raw,
        max_len=MAX_CHUNK_LEN,
        stride=STRIDE,
        q95_len_for_weak=q95_len_B
    )

    test_seq = split_long_sequences(
        test_seq_raw,
        max_len=MAX_CHUNK_LEN,
        stride=STRIDE,
        q95_len_for_weak=q95_len_B
    )

    val_seq["Split"] = "validation_B"
    test_seq["Split"] = "test_B"

    print("\nПосле chunking:")
    print("Validation_B raw:", val_seq_raw.shape, "->", val_seq.shape)
    print("Test_B raw:", test_seq_raw.shape, "->", test_seq.shape)

    print("\nМаксимальные длины после chunking:")
    print("Validation_B:", val_seq["SeqLen"].max())
    print("Test_B:", test_seq["SeqLen"].max())

    annotation_df = pd.concat([
        val_seq,
        test_seq
    ]).reset_index(drop=True)

    annotation_df["AnnotationID"] = range(len(annotation_df))

    annotation_df["StableID"] = (
        annotation_df["Split"].astype(str) + "_" +
        annotation_df["WindowStart"].astype(str) + "_" +
        annotation_df["WindowEnd"].astype(str) + "_" +
        annotation_df["ChunkID"].astype(str) + "_" +
        annotation_df["ChunkStartPos"].astype(str) + "_" +
        annotation_df["ChunkEndPos"].astype(str)
    )

    locked_data = {
        "params": {
            "VERSION": VERSION,
            "WINDOW_SIZE": WINDOW_SIZE,
            "MIN_SEQ_LEN": MIN_SEQ_LEN,
            "MAX_CHUNK_LEN": MAX_CHUNK_LEN,
            "STRIDE": STRIDE,
            "VAL_B_SIZE": VAL_B_SIZE,
            "TEST_B_SIZE": TEST_B_SIZE,
            "SERVER_NAME": SERVER_NAME,
            "SERVER_B_START_DATE": SERVER_B_START_DATE,
            "q95_len_B": q95_len_B,
            "HARD_CRITICAL_EVENT_IDS": list(HARD_CRITICAL_EVENT_IDS),
            "SOFT_SUSPICIOUS_EVENT_IDS": list(SOFT_SUSPICIOUS_EVENT_IDS),
        },
        "val_seq_raw": val_seq_raw,
        "test_seq_raw": test_seq_raw,
        "val_seq": val_seq,
        "test_seq": test_seq,
        "annotation_df": annotation_df,
        "template_to_id": template_to_id,
        "id_to_template": id_to_template
    }

    with open(LOCKED_DATA_PATH, "wb") as f:
        pickle.dump(locked_data, f)

    print("\nLocked Server B split saved:", LOCKED_DATA_PATH)

Locked Server B split не найден. Создаю validation_B / test_B...
Всего окон Server B: 1462
Validation_B raw: (731, 26)
Test_B raw: (731, 26)

Диапазоны времени:
Validation_B: 2026-04-04 21:00:00 — 2026-04-20 02:30:00
Test_B: 2026-04-20 02:30:00 — 2026-05-05 08:00:00

95-й перцентиль длины окон Server B: 36.0

После chunking:
Validation_B raw: (731, 26) -> (731, 31)
Test_B raw: (731, 26) -> (731, 31)

Максимальные длины после chunking:
Validation_B: 49
Test_B: 57

Locked Server B split saved: /content/drive/MyDrive/windows_logs_project/payment_serverB_locked_val_test_30min.pkl


In [ ]:
print("=" * 70)
print("SERVER B DATASET SIZES")
print("=" * 70)

print("val_seq_raw:", val_seq_raw.shape)
print("test_seq_raw:", test_seq_raw.shape)

print("\nAfter chunking:")
print("val_seq:", val_seq.shape)
print("test_seq:", test_seq.shape)
print("annotation_df:", annotation_df.shape)

print("\nTime ranges:")
print("validation_B:", val_seq["WindowStart"].min(), "—", val_seq["WindowEnd"].max())
print("test_B:", test_seq["WindowStart"].min(), "—", test_seq["WindowEnd"].max())

print("\nWeakLabel distribution:")
for name, df in [
    ("validation_B", val_seq),
    ("test_B", test_seq),
    ("annotation_df", annotation_df)
]:
    print(f"\n{name}:")
    print(df["WeakLabel"].value_counts(dropna=False))
    print(df["WeakLabel"].value_counts(normalize=True, dropna=False))

print("\nSeqLen stats:")
display(pd.DataFrame({
    "validation_B": val_seq["SeqLen"].describe(),
    "test_B": test_seq["SeqLen"].describe()
}))

SERVER B DATASET SIZES
val_seq_raw: (731, 26)
test_seq_raw: (731, 26)

After chunking:
val_seq: (731, 31)
test_seq: (731, 31)
annotation_df: (1462, 33)

Time ranges:
validation_B: 2026-04-04 21:00:00 — 2026-04-20 02:30:00
test_B: 2026-04-20 02:30:00 — 2026-05-05 08:00:00

WeakLabel distribution:

validation_B:
WeakLabel
0    515
1    216
Name: count, dtype: int64
WeakLabel
0    0.704514
1    0.295486
Name: proportion, dtype: float64

test_B:
WeakLabel
0    518
1    213
Name: count, dtype: int64
WeakLabel
0    0.708618
1    0.291382
Name: proportion, dtype: float64

annotation_df:
WeakLabel
0    1033
1     429
Name: count, dtype: int64
WeakLabel
0    0.706566
1    0.293434
Name: proportion, dtype: float64

SeqLen stats:


,validation_B,test_B
count,731.000000,731.000000
mean,23.288646,23.455540
std,7.073401,7.343743
min,9.000000,12.000000
25%,18.000000,18.000000
50%,20.000000,21.000000
75%,29.000000,29.000000
max,49.000000,57.000000


In [ ]:
def show_server_b_split_sizes():
    print("=" * 70)
    print("SERVER B DATASET SIZES")
    print("=" * 70)

    print("\nRaw windows:")
    print("val_seq_raw:  ", val_seq_raw.shape)
    print("test_seq_raw: ", test_seq_raw.shape)

    print("\nAfter chunking:")
    print("val_seq:      ", val_seq.shape)
    print("test_seq:     ", test_seq.shape)

    print("\nAnnotation:")
    print("annotation_df:", annotation_df.shape)

    print("\nTime ranges:")
    print("validation_B:", val_seq["WindowStart"].min(), "—", val_seq["WindowEnd"].max())
    print("test_B:      ", test_seq["WindowStart"].min(), "—", test_seq["WindowEnd"].max())

    print("\nSequence length stats:")
    length_stats = pd.DataFrame({
        "validation_B": val_seq["SeqLen"].describe(),
        "test_B": test_seq["SeqLen"].describe()
    })

    display(length_stats)

    print("\nWeakLabel distribution:")
    for name, df in [
        ("validation_B", val_seq),
        ("test_B", test_seq),
        ("annotation_df", annotation_df)
    ]:
        if "WeakLabel" in df.columns:
            print(f"\n{name}:")
            print(df["WeakLabel"].value_counts(dropna=False))
            print(df["WeakLabel"].value_counts(normalize=True, dropna=False))


show_server_b_split_sizes()

SERVER B DATASET SIZES

Raw windows:
val_seq_raw:   (731, 26)
test_seq_raw:  (731, 26)

After chunking:
val_seq:       (731, 31)
test_seq:      (731, 31)

Annotation:
annotation_df: (1462, 33)

Time ranges:
validation_B: 2026-04-04 21:00:00 — 2026-04-20 02:30:00
test_B:       2026-04-20 02:30:00 — 2026-05-05 08:00:00

Sequence length stats:


,validation_B,test_B
count,731.000000,731.000000
mean,23.288646,23.455540
std,7.073401,7.343743
min,9.000000,12.000000
25%,18.000000,18.000000
50%,20.000000,21.000000
75%,29.000000,29.000000
max,49.000000,57.000000



WeakLabel distribution:

validation_B:
WeakLabel
0    515
1    216
Name: count, dtype: int64
WeakLabel
0    0.704514
1    0.295486
Name: proportion, dtype: float64

test_B:
WeakLabel
0    518
1    213
Name: count, dtype: int64
WeakLabel
0    0.708618
1    0.291382
Name: proportion, dtype: float64

annotation_df:
WeakLabel
0    1033
1     429
Name: count, dtype: int64
WeakLabel
0    0.706566
1    0.293434
Name: proportion, dtype: float64


In [ ]:
print("Уникальных последовательностей:")
print("val_seq:     ", val_seq["Features"].apply(tuple).nunique())
print("test_seq:    ", test_seq["Features"].apply(tuple).nunique())

print("\nВсего строк:")
print("val_seq:     ", len(val_seq))
print("test_seq:    ", len(test_seq))

Уникальных последовательностей:
val_seq:      656
test_seq:     662

Всего строк:
val_seq:      731
test_seq:     731


In [ ]:
def shorten_text(text, max_chars=500):
    text = str(text).replace("\n", " ").replace("\r", " ")
    text = " ".join(text.split())

    if len(text) <= max_chars:
        return text

    return text[:max_chars] + " ...[truncated]"

In [ ]:
def build_annotation_prompt(row, max_events=80, max_message_chars=500):
    annotation_id = row["AnnotationID"]
    stable_id = row["StableID"]
    split = row["Split"]

    window_start = row["WindowStart"]
    window_end = row["WindowEnd"]
    seq_len = row["SeqLen"]

    levels = row.get("Levels", [])
    sources = row.get("Sources", [])
    event_ids = row.get("EventIDs", [])
    suspicious_reason = row.get("SuspiciousReason", "")

    features = row.get("Features", [])
    time_intervals = row.get("TimeInterval", [])
    review_templates = row.get("ReviewTemplates", [])
    raw_messages = row.get("RawMessages", [])

    event_lines = []

    n_events = min(len(features), max_events)

    for i in range(n_events):
        feature = features[i] if i < len(features) else ""
        delta_t = time_intervals[i] if i < len(time_intervals) else ""
        template = review_templates[i] if i < len(review_templates) else ""
        message = raw_messages[i] if i < len(raw_messages) else ""

        event_lines.append(
            f"{i + 1}. "
            f"Token: {feature} | "
            f"DeltaTimeSec: {delta_t} | "
            f"Template: {shorten_text(template, max_message_chars)} | "
            f"Message: {shorten_text(message, max_message_chars)}"
        )

    if len(features) > max_events:
        event_lines.append(
            f"... Остальные события не показаны: {len(features) - max_events}"
        )

    events_text = "\n".join(event_lines)

    prompt = f"""
Ты эксперт по анализу Windows Event Logs и обнаружению аномалий в последовательностях событий.

Нужно разметить одну последовательность событий из журналов Windows System/Application.

Классы:

0 = normal
Последовательность похожа на обычную активность системы: информационные события, обычный запуск/остановка служб, Windows Update, BITS, Group Policy, WMI/SCM без признаков сбоя.

1 = anomaly
Последовательность содержит признаки сбоя или нештатного поведения: Error/Critical, Windows Error Reporting, ошибки SQL Server, failed login, unexpected shutdown/restart, service terminated unexpectedly, audit log cleared, repeated crash reports, массовая остановка критичных служб или другая явно подозрительная цепочка.

Важно:
- Не считай Service Control Manager 7036 сам по себе аномалией.
- Не считай обычные циклы WMI Performance Adapter running/stopped аномалией без дополнительных признаков сбоя.
- Не считай Windows Update, BITS, Group Policy или AppX активность аномалией, если нет Error/Critical/WER/SQL-сбоев.
- DCOM timeout или Warning может быть аномалией только в контексте повторяемости, связи с ошибками или другими сбойными событиями.
- Если признаки слабые и нет явных ошибок, выбери label=0, но можешь поставить повышенный anomaly_score.
- Если есть повторяющиеся WER 1001 по sqlservr.exe, MSSQLSERVER Error, failed job SQL Agent, unexpected shutdown/restart или hard critical Event ID, выбери label=1.

Верни ответ строго в JSON формате без дополнительного текста:

{{
  "label": 0 или 1,
  "anomaly_score": число от 0.0 до 1.0,
  "confidence": число от 0.0 до 1.0,
  "key_events": ["список наиболее важных Event ID / источников / токенов"],
  "reason": "краткое объяснение на русском языке"
}}

Метаданные последовательности:
AnnotationID: {annotation_id}
StableID: {stable_id}
Split: {split}
WindowStart: {window_start}
WindowEnd: {window_end}
SeqLen: {seq_len}

Levels в chunk-е:
{levels}

Sources в chunk-е:
{sources}

EventIDs в chunk-е:
{event_ids}

Предварительные weak-label причины chunk-а:
{suspicious_reason}

События последовательности:
{events_text}
""".strip()

    return prompt

In [ ]:
annotation_prompts = annotation_df.copy()

annotation_prompts["Prompt"] = annotation_prompts.apply(
    lambda row: build_annotation_prompt(
        row,
        max_events=80,
        max_message_chars=500
    ),
    axis=1
)

prompt_columns = [
    "AnnotationID",
    "StableID",
    "Split",
    "WindowStart",
    "WindowEnd",
    "SeqLen",
    "Levels",
    "Sources",
    "EventIDs",
    "SuspiciousReason",
    "Prompt"
]

annotation_prompts_export = annotation_prompts[prompt_columns].copy()

list_cols = [
    "Levels",
    "Sources",
    "EventIDs"
]

for col in list_cols:
    annotation_prompts_export[col] = annotation_prompts_export[col].apply(
        lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, list) else x
    )

ANNOTATION_PROMPTS_CSV_PATH = f"{PROMPT_EXPORT_DIR}/windows_val_test_annotation_prompts_30min_v3.csv"
ANNOTATION_PROMPTS_JSONL_PATH = f"{PROMPT_EXPORT_DIR}/windows_val_test_annotation_prompts_30min_v3.jsonl"

annotation_prompts_export.to_csv(
    ANNOTATION_PROMPTS_CSV_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("CSV saved:", ANNOTATION_PROMPTS_CSV_PATH)
print("Shape:", annotation_prompts_export.shape)

display(annotation_prompts_export.head())

CSV saved: /content/drive/MyDrive/windows_logs_project/payment_serverB_annotation_prompts_30min/windows_val_test_annotation_prompts_30min_v3.csv
Shape: (1462, 11)


,AnnotationID,StableID,Split,WindowStart,WindowEnd,SeqLen,Levels,Sources,EventIDs,SuspiciousReason,Prompt
0,0,validation_B_2026-04-04 21:00:00_2026-04-04 21...,validation_B,2026-04-04 21:00:00,2026-04-04 21:30:00,9,"[""Information""]","[""Service Control Manager""]","[""7036"", ""7040""]",,Ты эксперт по анализу Windows Event Logs и обн...
1,1,validation_B_2026-04-04 21:30:00_2026-04-04 22...,validation_B,2026-04-04 21:30:00,2026-04-04 22:00:00,24,"[""Information"", ""Warning""]","[""Microsoft-Windows-GroupPolicy"", ""SceCli"", ""S...","[""1202"", ""1502"", ""7036"", ""7040""]",soft_suspicious_event_id:1202,Ты эксперт по анализу Windows Event Logs и обн...
2,2,validation_B_2026-04-04 22:00:00_2026-04-04 22...,validation_B,2026-04-04 22:00:00,2026-04-04 22:30:00,17,"[""Information""]","[""Service Control Manager""]","[""7036"", ""7040""]",,Ты эксперт по анализу Windows Event Logs и обн...
3,3,validation_B_2026-04-04 22:30:00_2026-04-04 23...,validation_B,2026-04-04 22:30:00,2026-04-04 23:00:00,18,"[""Information""]","[""Service Control Manager""]","[""7036"", ""7040""]",,Ты эксперт по анализу Windows Event Logs и обн...
4,4,validation_B_2026-04-04 23:00:00_2026-04-04 23...,validation_B,2026-04-04 23:00:00,2026-04-04 23:30:00,26,"[""Information"", ""Warning""]","[""Microsoft-Windows-GroupPolicy"", ""SceCli"", ""S...","[""1202"", ""1502"", ""7036"", ""7040""]",soft_suspicious_event_id:1202,Ты эксперт по анализу Windows Event Logs и обн...


In [ ]:
with open(ANNOTATION_PROMPTS_JSONL_PATH, "w", encoding="utf-8") as f:
    for _, row in annotation_prompts.iterrows():
        record = {
            "AnnotationID": int(row["AnnotationID"]),
            "StableID": str(row["StableID"]),
            "Split": str(row["Split"]),
            "WindowStart": str(row["WindowStart"]),
            "WindowEnd": str(row["WindowEnd"]),
            "SeqLen": int(row["SeqLen"]),
            "Prompt": row["Prompt"]
        }

        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("JSONL saved:", ANNOTATION_PROMPTS_JSONL_PATH)

JSONL saved: /content/drive/MyDrive/windows_logs_project/payment_serverB_annotation_prompts_30min/windows_val_test_annotation_prompts_30min_v3.jsonl


In [ ]:
print(annotation_prompts.loc[0, "Prompt"])

Ты эксперт по анализу Windows Event Logs и обнаружению аномалий в последовательностях событий.

Нужно разметить одну последовательность событий из журналов Windows System/Application.

Классы:

0 = normal
Последовательность похожа на обычную активность системы: информационные события, обычный запуск/остановка служб, Windows Update, BITS, Group Policy, WMI/SCM без признаков сбоя.

1 = anomaly
Последовательность содержит признаки сбоя или нештатного поведения: Error/Critical, Windows Error Reporting, ошибки SQL Server, failed login, unexpected shutdown/restart, service terminated unexpectedly, audit log cleared, repeated crash reports, массовая остановка критичных служб или другая явно подозрительная цепочка.

Важно:
- Не считай Service Control Manager 7036 сам по себе аномалией.
- Не считай обычные циклы WMI Performance Adapter running/stopped аномалией без дополнительных признаков сбоя.
- Не считай Windows Update, BITS, Group Policy или AppX активность аномалией, если нет Error/Critica

In [ ]:
ANNOTATION_RESULTS_TEMPLATE_PATH = f"{PROMPT_EXPORT_DIR}/windows_val_test_annotation_results_template_binary_30min_v3.csv"

annotation_results_template = annotation_prompts[[
    "AnnotationID",
    "StableID",
    "Split",
    "WindowStart",
    "WindowEnd",
    "SeqLen"
]].copy()

annotation_results_template["label"] = None
annotation_results_template["anomaly_score"] = None
annotation_results_template["confidence"] = None
annotation_results_template["key_events"] = None
annotation_results_template["reason"] = None
annotation_results_template["annotator"] = None

annotation_results_template.to_csv(
    ANNOTATION_RESULTS_TEMPLATE_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Binary annotation results template saved:", ANNOTATION_RESULTS_TEMPLATE_PATH)

display(annotation_results_template.head())

Binary annotation results template saved: /content/drive/MyDrive/windows_logs_project/payment_serverB_annotation_prompts_30min/windows_val_test_annotation_results_template_binary_30min_v3.csv


,AnnotationID,StableID,Split,WindowStart,WindowEnd,SeqLen,label,anomaly_score,confidence,key_events,reason,annotator
0,0,validation_B_2026-04-04 21:00:00_2026-04-04 21...,validation_B,2026-04-04 21:00:00,2026-04-04 21:30:00,9,None,None,None,None,None,None
1,1,validation_B_2026-04-04 21:30:00_2026-04-04 22...,validation_B,2026-04-04 21:30:00,2026-04-04 22:00:00,24,None,None,None,None,None,None
2,2,validation_B_2026-04-04 22:00:00_2026-04-04 22...,validation_B,2026-04-04 22:00:00,2026-04-04 22:30:00,17,None,None,None,None,None,None
3,3,validation_B_2026-04-04 22:30:00_2026-04-04 23...,validation_B,2026-04-04 22:30:00,2026-04-04 23:00:00,18,None,None,None,None,None,None
4,4,validation_B_2026-04-04 23:00:00_2026-04-04 23...,validation_B,2026-04-04 23:00:00,2026-04-04 23:30:00,26,None,None,None,None,None,None


In [ ]:

val_seq.to_pickle(f"{EXPORT_DIR}/val_seq.pkl")
test_seq.to_pickle(f"{EXPORT_DIR}/test_seq.pkl")


val_seq_raw.to_pickle(f"{EXPORT_DIR}/val_seq_raw.pkl")
test_seq_raw.to_pickle(f"{EXPORT_DIR}/test_seq_raw.pkl")

annotation_df.to_pickle(f"{EXPORT_DIR}/annotation_df.pkl")

with open(f"{EXPORT_DIR}/template_to_id.pkl", "wb") as f:
    pickle.dump(template_to_id, f)

with open(f"{EXPORT_DIR}/id_to_template.pkl", "wb") as f:
    pickle.dump(id_to_template, f)

with open(f"{EXPORT_DIR}/template_to_id.json", "w", encoding="utf-8") as f:
    json.dump(template_to_id, f, ensure_ascii=False, indent=2)

with open(f"{EXPORT_DIR}/id_to_template.json", "w", encoding="utf-8") as f:
    json.dump(id_to_template, f, ensure_ascii=False, indent=2)

print("Pickle and dictionary files saved.")

Pickle and dictionary files saved.


In [ ]:
def save_df_csv_json_lists(df: pd.DataFrame, path: str):
    df_to_save = df.copy()

    list_cols = [
        "Features",
        "TimeInterval",
        "LogNames",
        "Sources",
        "Levels",
        "EventIDs",
        "RawMessages",
        "ReviewTemplates",
        "EventLogNames",
        "EventSources",
        "EventLevels",
        "EventIDsPerEvent",
        "EventTokens"
    ]

    for col in list_cols:
        if col in df_to_save.columns:
            df_to_save[col] = df_to_save[col].apply(
                lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, list) else x
            )

    df_to_save.to_csv(path, index=False, encoding="utf-8-sig")

In [ ]:

save_df_csv_json_lists(val_seq, f"{EXPORT_DIR}/val_seq_B.csv")
save_df_csv_json_lists(test_seq, f"{EXPORT_DIR}/test_seqB.csv")
save_df_csv_json_lists(annotation_df, f"{EXPORT_DIR}/annotation_dfB.csv")

print("CSV files saved.")

CSV files saved.


In [ ]:
LOCAL_TRAINING_DATA_PATH = f"{EXPORT_DIR}/windows_training_data_30min_v3.pkl"

local_training_data = {
    "params": {
        "VERSION": VERSION,
        "WINDOW_SIZE": WINDOW_SIZE,
        "MIN_SEQ_LEN": MIN_SEQ_LEN,
        "MAX_CHUNK_LEN": MAX_CHUNK_LEN,
        "STRIDE": STRIDE,
        "TRAIN_SIZE": TRAIN_SIZE,
        "VAL_SIZE": VAL_SIZE,
        "TEST_SIZE": TEST_SIZE,
    },
    "train_normal": train_normal,
    "val_seq": val_seq,
    "test_seq": test_seq,
    "annotation_df": annotation_df,
    "template_to_id": template_to_id,
    "id_to_template": id_to_template
}

with open(LOCAL_TRAINING_DATA_PATH, "wb") as f:
    pickle.dump(local_training_data, f)

print("Local training data saved:", LOCAL_TRAINING_DATA_PATH)

In [ ]:
with open(LOCAL_TRAINING_DATA_PATH, "rb") as f:
    check_data = pickle.load(f)

print(check_data.keys())
print("train_normal:", check_data["train_normal"].shape)
print("val_seq:", check_data["val_seq"].shape)
print("test_seq:", check_data["test_seq"].shape)
print("annotation_df:", check_data["annotation_df"].shape)

In [ ]:
THREE_LLM_TEMPLATE_PATH = (
    f"{PROMPT_EXPORT_DIR}/windows_val_test_annotation_results_3llm_30min_v3B.csv"
)

annotation_results_3llm = annotation_prompts[[
    "AnnotationID",
    "StableID",
    "Split",
    "WindowStart",
    "WindowEnd",
    "SeqLen"
]].copy()

for model_name in ["llm1", "llm2", "llm3"]:
    annotation_results_3llm[f"{model_name}_label"] = None
    annotation_results_3llm[f"{model_name}_score"] = None
    annotation_results_3llm[f"{model_name}_confidence"] = None
    annotation_results_3llm[f"{model_name}_key_events"] = None
    annotation_results_3llm[f"{model_name}_reason"] = None
    annotation_results_3llm[f"{model_name}_annotator"] = None

annotation_results_3llm.to_csv(
    THREE_LLM_TEMPLATE_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("3-LLM annotation template saved:", THREE_LLM_TEMPLATE_PATH)
display(annotation_results_3llm.head())

3-LLM annotation template saved: /content/drive/MyDrive/windows_logs_project/payment_serverB_annotation_prompts_30min/windows_val_test_annotation_results_3llm_30min_v3B.csv


,AnnotationID,StableID,Split,WindowStart,WindowEnd,SeqLen,llm1_label,llm1_score,llm1_confidence,llm1_key_events,...,llm2_confidence,llm2_key_events,llm2_reason,llm2_annotator,llm3_label,llm3_score,llm3_confidence,llm3_key_events,llm3_reason,llm3_annotator
0,0,validation_B_2026-04-04 21:00:00_2026-04-04 21...,validation_B,2026-04-04 21:00:00,2026-04-04 21:30:00,9,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
1,1,validation_B_2026-04-04 21:30:00_2026-04-04 22...,validation_B,2026-04-04 21:30:00,2026-04-04 22:00:00,24,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2,2,validation_B_2026-04-04 22:00:00_2026-04-04 22...,validation_B,2026-04-04 22:00:00,2026-04-04 22:30:00,17,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
3,3,validation_B_2026-04-04 22:30:00_2026-04-04 23...,validation_B,2026-04-04 22:30:00,2026-04-04 23:00:00,18,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
4,4,validation_B_2026-04-04 23:00:00_2026-04-04 23...,validation_B,2026-04-04 23:00:00,2026-04-04 23:30:00,26,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
